# Frontier Vision-Language Models Fail on Urdu Nastaliq
## Evaluation of Qwen3-VL-2B-Instruct on Authentic Pakistani Restaurant Menus

**Technical Evaluation: Blind Spots of Frontier Vision-Language Models**

This notebook evaluates how frontier open-weight vision-language models (VLMs) handle authentic Pakistani restaurant menus containing side-by-side English (Latin) and Urdu (Nastaliq) text. 

Using [**Qwen3-VL-2B-Instruct**](https://huggingface.co/Qwen/Qwen3-VL-2B-Instruct) on 5 menu pages (60 annotated items) from **Noorani Kabab House**, we measure optical character recognition and layout understanding across both scripts. Across the 58 core restaurant dishes (pages 2 to 5), the model achieves **98.3% exact match accuracy** on English dish names (with **100% price extraction accuracy**), but drops to **1.7%** on Urdu Nastaliq, with a Character Error Rate (CER) that is **590.1x higher** (0.5901 vs. 0.0010). Across the entire 60-item dataset including cover notices, the CER disparity is 582.5x (0.5825 vs. 0.0010).

## 1. The Typographic Challenge: Nastaliq vs. Naskh

Standard Arabic is written in the **Naskh** script. Naskh uses a flat horizontal baseline where letters join sequentially from right to left. Because letter heights and baselines remain uniform, modern digital layout engines and visual tokenizers process Naskh with predictable spatial relationships.

Urdu is traditionally written in the **Nastaliq** calligraphic style. Nastaliq differs from Naskh in four ways:
1. **Sloping Baselines**: Each word or sub-word cluster forms along a diagonal slant (roughly 30 to 45 degrees), cascading downward from upper right to lower left.
2. **Stacked Vertical Ligatures**: Characters stack on top of one another rather than chaining horizontally. A single word can contain multiple vertical tiers.
3. **Context-Sensitive Glyph Variants**: The visual shape of an Urdu letter changes based on its position within a stacked ligature.
4. **Expanded Alphabet**: Urdu expands the 28-letter Arabic alphabet to 39 basic letters (adding retroflex sounds like ٹ, ڈ, ڑ) plus 14 aspirated compound consonants (such as جھ, گھ), creating a substantially larger glyph and ligature inventory.

![Nastaliq vs Naskh](concepts/naskh%20vs%20nastaliq.jpg)

## 2. Computational Failure Mechanism in Vision-Language Models

Qwen3-VL processes images using a native-resolution Vision Transformer with 16×16 pixel patches (`patch_size: 16`) and 2D Multimodal Rotary Position Embeddings (M-RoPE). 

While self-attention allows the vision encoder to reconstruct standard horizontal Latin and Naskh text across adjacent patches, Urdu Nastaliq presents a compounding architectural and data challenge:

1. **Orthogonal Coordinate Mismatch**: M-RoPE and standard ViT attention mechanisms compute spatial relations along Cartesian axes (horizontal rows and vertical columns). In Nastaliq, semantic continuity follows a diagonal cascading vector (30° to 45°). A single stacked ligature spans multiple vertical tiers within a single word cluster, breaking the horizontal scanning bias common in vision-language decoders.
2. **Pretraining Data Imbalance**: Frontier vision encoders are pre-trained on billions of Latin and standard Arabic Naskh document images from web crawls, but virtually zero high-quality paired Nastaliq datasets. Visual feature embeddings for unrepresented Nastaliq ligatures carry high entropy and low activation confidence.
3. **Language Model Prior Dominance**: When visual features from the Urdu column are uncertain, the decoder falls back on its strongest available signal: the adjacent English text. This produces three distinct failure behaviors:
   - **Ligature Fracturing**: Misidentifying stacked character clusters, splitting joined letters into disconnected Arabic graphemes or turning Urdu letters into unrelated words.
   - **Phonetic Back-Transliteration**: Reading the English column and spelling the English word phonetically in Arabic letters (*مکھن* → *بٹر*).
   - **RTL-LTR Column Alignment Trap**: Confronted with a hybrid layout (English on the left, Urdu on the right, numbers in the center), the attention mechanism drops into repetition loops over price digits.

![VLM Nastaliq Failure Modes](concepts/vlm_nastaliq_failure.jpg)

In [1]:
import os
import sys
import json
import time
import warnings
import torch
from PIL import Image
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor

warnings.filterwarnings('ignore')

# Ensure UTF-8 output encoding for non-ASCII text
if sys.platform == "win32" and hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")

# Check GPU acceleration
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Active Device: {device}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"Allocated VRAM: {torch.cuda.memory_allocated() / (1024**2):.1f} MB")


c:\Users\Hp\Desktop\Stuff\Clout\Tennis\Fatima Fellowship\blind-spots\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Active Device: cuda
GPU Name: NVIDIA GeForce RTX 4060 Laptop GPU
Allocated VRAM: 0.0 MB


In [2]:
# Resolve model location: prioritize local folder, fall back to Hugging Face Hub
local_path = os.path.join("models", "Qwen3-VL-2B-Instruct")
model_id = local_path if (os.path.exists(local_path) and os.path.exists(os.path.join(local_path, "model.safetensors"))) else "Qwen/Qwen3-VL-2B-Instruct"

print(f"Loading processor from: {model_id}")
t0 = time.perf_counter()
processor = AutoProcessor.from_pretrained(model_id)
t_proc = time.perf_counter() - t0
print(f"Processor ready in {t_proc:.2f}s")

print(f"Loading Qwen3-VL-2B-Instruct weights into {device}...")
t0 = time.perf_counter()
model = Qwen3VLForConditionalGeneration.from_pretrained(
    model_id,
    dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None
)
t_model = time.perf_counter() - t0
print(f"Model loaded in {t_model:.2f}s")
if torch.cuda.is_available():
    print(f"VRAM after loading: {torch.cuda.memory_allocated() / (1024**2):.1f} MB")


Loading processor from: models\Qwen3-VL-2B-Instruct
Processor ready in 0.27s
Loading Qwen3-VL-2B-Instruct weights into cuda...


Loading weights: 100%|██████████| 625/625 [00:02<00:00, 278.37it/s]

Model loaded in 3.66s
VRAM after loading: 4058.0 MB


## 3. Visual Inspection of Menu Crops

Here are two representative crops from the test dataset showing the failure patterns in practice:

### A. Ligature Fracturing (Page 1 Notice)
Ground truth text: **ہماری کوئی اور برانچ نہیں ہے** ("We have no other branch").
The model outputs: **جاری کوئی اور برائے نہیں ہے**.
The initial word **ہماری** has a stacked ligature where the letter *ہ* sits directly above *م*. The model misclassifies this as **جاری**. Similarly, the word **برانچ** (*branch*) contains a nested *ن* above *چ*, which the model fractures into **برائے**.

![Cover Notice Crop](concepts/notice_crop.jpg)

### B. Phonetic Back-Transliteration (Page 2 Specialties)
Ground truth Urdu text for butter variants: **(مکھن کے ساتھ)** (*Makhan ke sath* - with butter).
The model outputs: **(بٹر)** (*Batar* - phonetic English "Butter").
Rather than decoding the Nastaliq characters for *مکھن*, the vision model attends to the English label `(WITH BUTTER)` on the opposite side of the row and writes the phonetic English word in Arabic script.

![Kata Kat Menu Crop](concepts/katakat_crop.jpg)

In [3]:
# Run interactive inference on a sample menu page
sample_image_path = os.path.join("dataset", "menu", "2.jpg")
sample_img = Image.open(sample_image_path).convert("RGB")
print(f"Loaded menu page: {sample_image_path} ({sample_img.width}x{sample_img.height} px)")

def run_vlm_query(image, prompt_text, max_tokens=1024):
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": prompt_text}
            ]
        }
    ]
    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt"
    ).to(model.device)

    t0 = time.perf_counter()
    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            do_sample=False
        )
    gen_time = time.perf_counter() - t0
    
    trimmed_ids = [out[len(inp):] for inp, out in zip(inputs.input_ids, generated_ids)]
    decoded_text = processor.batch_decode(trimmed_ids, skip_special_tokens=True)[0]
    tok_per_sec = len(trimmed_ids[0]) / gen_time if gen_time > 0 else 0
    return decoded_text.strip(), gen_time, tok_per_sec

# 1. English extraction test
print("--- [Test 1] English Extraction ---")
en_out, en_time, en_speed = run_vlm_query(
    sample_img, 
    "Extract all English dish names and their prices from this menu. Format as a clean list: 'Dish Name - Price'."
)
print(f"Generated in {en_time:.2f}s ({en_speed:.1f} tok/s):")
for line in en_out.split("\n")[:6]:
    print(f"  {line}")

# 2. Urdu extraction test
print("\n--- [Test 2] Urdu Extraction ---")
ur_out, ur_time, ur_speed = run_vlm_query(
    sample_img, 
    "Extract all Urdu dish names and their prices from this menu. Format as a clean list: 'اردو نام - قیمت'."
)
print(f"Generated in {ur_time:.2f}s ({ur_speed:.1f} tok/s):")
for line in ur_out.split("\n")[:6]:
    print(f"  {line}")


Loaded menu page: dataset\menu\2.jpg (638x800 px)
--- [Test 1] English Extraction ---
Generated in 11.75s (21.1 tok/s):
  KATA KAT MIX PLATE - 1430
  KATA KAT MIX PLATE (WITH BUTTER) - 1630
  BRAIN MASALA - 1100
  BRAIN MASALA (WITH BUTTER) - 1300
  KIDNEY MASALA - 880
  KIDNEY MASALA (WITH BUTTER) - 1080

--- [Test 2] Urdu Extraction ---
Generated in 13.25s (23.7 tok/s):
  - کاٹ کاٹ میکس پلیٹ - 1430
  - کاٹ کاٹ میکس پلیٹ (بٹر) - 1630
  - براين ماسلا - 1100
  - براين ماسلا (بٹر) - 1300
  - کیدی ماسلا - 880
  - کیدی ماسلا (بٹر) - 1080


In [4]:
# Live Levenshtein distance metrics computation and verification
def levenshtein_dist(s1, s2):
    m, n = len(s1), len(s2)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            cost = 0 if s1[i - 1] == s2[j - 1] else 1
            dp[i][j] = min(dp[i - 1][j] + 1, dp[i][j - 1] + 1, dp[i - 1][j - 1] + cost)
    return dp[m][n]

def calculate_cer(ref, hyp):
    r, h = ref.strip(), hyp.strip()
    return (levenshtein_dist(r, h) / len(r)) if r else (0.0 if not h else 1.0)

def calculate_wer(ref, hyp):
    rw, hw = ref.strip().split(), hyp.strip().split()
    if not rw:
        return 0.0 if not hw else 1.0
    m, n = len(rw), len(hw)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            cost = 0 if rw[i - 1] == hw[j - 1] else 1
            dp[i][j] = min(dp[i - 1][j] + 1, dp[i][j - 1] + 1, dp[i - 1][j - 1] + cost)
    return dp[m][n] / len(rw)

# Verify metrics live from results/benchmark_scores.json
benchmark_file = os.path.join("results", "benchmark_scores.json")
with open(benchmark_file, "r", encoding="utf-8") as f:
    bench_data = json.load(f)

# Re-compute CER live across sample items to verify calculation functions
live_verified_count = 0
for item in bench_data.get("sample_item_comparisons", []):
    gt_en, pred_en = item["ground_truth_en"], item["predicted_en"]
    gt_ur, pred_ur = item["ground_truth_ur"], item["predicted_ur"]
    if gt_en and pred_en:
        cer_en_calc = calculate_cer(gt_en.lower(), pred_en.lower())
        assert abs(cer_en_calc - item["cer_en"]) < 0.05
    if gt_ur and pred_ur:
        cer_ur_calc = calculate_cer(gt_ur.lower(), pred_ur.lower())
        assert abs(cer_ur_calc - item["cer_ur"]) < 0.05
    live_verified_count += 1

cm = bench_data["core_menu_dishes_metrics"]
print("=" * 70)
print("      QUANTITATIVE BENCHMARK: ENGLISH VS URDU NASTALIQ")
print("=" * 70)
print(f"Model                      : {bench_data['model']}")
print(f"Core Menu Dishes Evaluated : {cm['items_count']['english']} (Pages 2-5)")
print(f"Priced Items Verified      : {cm['items_count']['priced_items_checked']}/{cm['items_count']['english']} ({cm['english']['price_accuracy_pct']:.1f}% match)")
print(f"Live Sample Calculations   : {live_verified_count} verified")
print("-" * 70)
print("Metric                       English (Latin)    Urdu (Nastaliq)")
print("-" * 70)
print(f"Exact Match Accuracy       :   {cm['english']['exact_match_accuracy_pct']:>5.1f}%             {cm['urdu_nastaliq']['exact_match_accuracy_pct']:>5.1f}%")
print(f"Character Error Rate (CER) :   {cm['english']['character_error_rate_cer']:>6.4f}             {cm['urdu_nastaliq']['character_error_rate_cer']:>6.4f}")
print(f"Word Error Rate (WER)      :   {cm['english']['word_error_rate_wer']:>6.4f}             {cm['urdu_nastaliq']['word_error_rate_wer']:>6.4f}")
print(f"Price Accuracy             :   {cm['english']['price_accuracy_pct']:>5.1f}%                 N/A")
print("-" * 70)
print(f"Disparity Ratio: Urdu CER is {cm['disparity_ratio']['cer_ratio_urdu_vs_english']}x higher than English.")
print(f"Accuracy Drop  : {cm['disparity_ratio']['accuracy_drop_pct']}% absolute drop on core menu dishes.")
print("=" * 70)

fm = bench_data["full_dataset_metrics"]
print(f"\nFull Dataset Scope (Pages 1-5, {fm['items_count']['english']} total items including cover notices):")
print(f"  English CER: {fm['english']['character_error_rate_cer']:.4f} | Urdu CER: {fm['urdu_nastaliq']['character_error_rate_cer']:.4f} | Disparity: {fm['disparity_ratio']['cer_ratio_urdu_vs_english']}x")
print(f"  English Exact Match: {fm['english']['exact_match_accuracy_pct']:.1f}% | Urdu Exact Match: {fm['urdu_nastaliq']['exact_match_accuracy_pct']:.1f}%")


      QUANTITATIVE BENCHMARK: ENGLISH VS URDU NASTALIQ
Model                      : Qwen/Qwen3-VL-2B-Instruct
Core Menu Dishes Evaluated : 58 (Pages 2-5)
Priced Items Verified      : 58/58 (100.0% match)
Live Sample Calculations   : 10 verified
----------------------------------------------------------------------
Metric                       English (Latin)    Urdu (Nastaliq)
----------------------------------------------------------------------
Exact Match Accuracy       :    98.3%               1.7%
Character Error Rate (CER) :   0.0010             0.5901
Word Error Rate (WER)      :   0.0057             0.9368
Price Accuracy             :   100.0%                 N/A
----------------------------------------------------------------------
Disparity Ratio: Urdu CER is 590.1x higher than English.
Accuracy Drop  : 96.55% absolute drop on core menu dishes.

Full Dataset Scope (Pages 1-5, 60 total items including cover notices):
  English CER: 0.0010 | Urdu CER: 0.5825 | Disparity: 582.5x

## 4. Quantitative Results & Comparison

Evaluating across the menu pages reveals a steep performance cliff between Latin script English and Nastaliq script Urdu:

### Primary Evaluation: Core Menu Dishes (Pages 2 to 5, 58 items)

| Evaluation Metric | English Column (Latin Script) | Urdu Column (Nastaliq Script) | Disparity Ratio |
|---|---|---|---|
| **Exact Match Accuracy** | **98.3%** (57/58) | **1.7%** (1/58) | -96.6% absolute drop |
| **Character Error Rate (CER)** | **0.0010** | **0.5901** | **590.1x higher error** |
| **Word Error Rate (WER)** | **0.0057** | **0.9368** | **164.4x higher error** |
| **Price Extraction Accuracy** | **100.0%** (58/58) | N/A | Full dual-price coverage |

*Note: Across the full 60-item dataset (including the 2 non-priced cover notice items from page 1), English exact match is 98.3% (CER 0.0010) and Urdu exact match is 1.7% (CER 0.5825), maintaining a 582.5x error disparity.*

### Item-Level Comparison Samples

| Dish | Ground Truth Urdu | Qwen3-VL Output | Failure Type |
|---|---|---|---|
| Brain Masala (Butter) | برین مصالحہ (مکھن کے ساتھ) | براين ماسلا (بٹر) | Transliteration (*مکھن* → *بٹر*) |
| Mutton Champs Masala | مٹن چانپ مصالحہ | موتون چامپس ماسلا | Transliteration (*چانپ* → *چامپس*) |
| Chapati | چپاتی | شاپاتی | Consonant substitution (*چ* → *ش*) |
| Cover Notice | ہماری کوئی اور برانچ نہیں ہے | جاری کوئی اور برائے نہیں ہے | Ligature fracturing (*ہماری* → *جاری*) |
| Peshawari Karahi | مٹن پشاوری کڑاہی | مUTTON / پESHAWARI کاراھی | Script mixing & transliteration (*مٹن* → *مUTTON*) |

## 5. Research Significance & Impact

### The Benchmark Gap: Official Claims vs. Real-World Failure
The official Qwen3-VL model card and technical report (Qwen Team, 2025) report state-of-the-art multimodal document understanding for **Qwen3-VL-2B-Instruct**, citing:
- **DocVQA**: **93.3%** accuracy
- **OCRBench**: **858 / 1000**
- **CC-OCR-Bench**: **72.8%** overall score
- **Expanded OCR Coverage**: Advertised support across 32 languages

Standard OCR and VLM evaluation suites rely on flat horizontal baselines typical of Latin, Chinese, and Arabic Naskh script. When evaluated on real-world Pakistani restaurant menus in **Nastaliq**, the exact match accuracy of Qwen3-VL-2B-Instruct drops to **1.7%**, with Character Error Rate increasing by **590.1x** relative to English on the same document pages (0.5901 vs. 0.0010).

### Architectural Grounding: Interleaved-MRoPE & DeepStack
Qwen3-VL's architecture uses two core mechanisms:
1. **Interleaved-MRoPE**: Allocates positional embeddings across 2D Cartesian grid coordinates (height and width).
2. **DeepStack**: Fuses multi-level Vision Transformer (ViT) patch features directly into language decoder blocks.

In Nastaliq typography, words form along diagonal cascading baselines (30° to 45°) with characters stacked in vertical tiers. Rigid 16×16 pixel Cartesian patches split connected ligatures across disparate visual tokens. Because the resulting visual embeddings carry high entropy, the decoder defaults to its strongest available language prior: reading the English column and writing phonetic approximations in Arabic letters (*مکھن* → *بٹر*).

### Why This Matters for South Asian AI
Urdu is spoken by over 230 million people worldwide and is the national language of Pakistan. Across newspapers, legal records, packaging, and commercial signage, Urdu is written almost exclusively in Nastaliq calligraphy rather than Arabic Naskh. Benchmarks that report strong Arabic OCR create a false impression of capability, leaving a major linguistic community unserved by frontier multimodal models.

### Open Research Directions
1. **Curvilinear Visual Tokenization**: Designing vision encoders that follow non-Cartesian, diagonal cursive trajectories instead of rigid rectangular grids.
2. **Nastaliq Synthetic Alignment**: Incorporating high-resolution paired Nastaliq corpora into multimodal projection training.
3. **Cross-Script Attention Regularization**: Constraining cross-attention layers to prevent visual leakage from adjacent Latin text when transcribing vernacular scripts.

### References
- Model Card: [Qwen/Qwen3-VL-2B-Instruct (Hugging Face)](https://huggingface.co/Qwen/Qwen3-VL-2B-Instruct)
- Technical Report: Qwen Team (2025). *Qwen3 Technical Report*. [arXiv:2505.09388 [cs.CL]](https://arxiv.org/abs/2505.09388).